# Testing Notebook

# Enformer

In [ ]:
import torch
from enformer_pytorch import Enformer, seq_indices_to_one_hot

model = Enformer.from_hparams(
    dim=1536,
    depth=11,
    heads=8,
    output_heads=dict(human=5313, mouse=1643),  # noqa: C408
    target_length=896,
)

seq = torch.randint(0, 5, (1, 196_608))
one_hot = seq_indices_to_one_hot(seq)

output, embeddings = model(one_hot, return_embeddings=True)

embeddings  # (1, 896, 3072)

tensor([[[ 0.2573,  0.1184,  0.3862,  ...,  0.2613, -0.0000,  0.0315],
         [ 0.0012,  0.4613,  0.4624,  ..., -0.0172,  0.0690,  0.1773],
         [-0.0000,  0.0455,  0.3596,  ...,  0.1237, -0.0104,  0.0446],
         ...,
         [-0.1636,  0.1695,  0.0000,  ...,  0.0493,  0.1665, -0.0124],
         [ 0.0872,  0.1877,  0.5801,  ...,  0.3235, -0.0951, -0.0811],
         [ 0.1893, -0.0970,  0.0843,  ...,  0.2897,  0.3593,  0.2840]]],
       grad_fn=<MulBackward0>)

In [7]:
print(one_hot.shape)

torch.Size([1, 196608, 4])


In [ ]:
# In [1]: Imports and setup
import random
import torch

from embpy.models.dna_models import EnformerWrapper

# 1) Initialize the Enformer wrapper on CPU
wrapper = EnformerWrapper("EleutherAI/enformer-official-rough")
wrapper.load(device=torch.device("cpu"))
print("Enformer model loaded. Sequence length =", wrapper.SEQUENCE_LENGTH)


# 2) Helper to generate a random DNA string of exactly SEQUENCE_LENGTH
def random_dna_fixed(length: int) -> str:  # noqa: D103
    return "".join(random.choice("ACGTN") for _ in range(length))


# 3) Build a small batch of random sequences
batch_size = 4
seq_len = wrapper.SEQUENCE_LENGTH
dna_batch = [random_dna_fixed(seq_len) for _ in range(batch_size)]
print("Batch sizes:", [len(s) for s in dna_batch])

# 4) Run embed_batch with mean pooling
embeddings = wrapper.embed_batch(dna_batch, pooling_strategy="mean")

print("\nembed_batch outputs:")
for i, emb in enumerate(embeddings):
    print(f"  Sequence {i}: shape {emb.shape}, first 5 dims {emb[:5]}")

Enformer model loaded. Sequence length = 196608
Batch sizes: [196608, 196608, 196608, 196608]

embed_batch outputs:
  Sequence 0: shape (3072,), first 5 dims [-0.09349184 -0.00098717 -0.00434084 -0.09515201  0.21379483]
  Sequence 1: shape (3072,), first 5 dims [-0.12184776 -0.00174527  0.0444397  -0.06221178  0.24697961]
  Sequence 2: shape (3072,), first 5 dims [-0.11219615 -0.00108458  0.07503723 -0.03661817  0.17436083]
  Sequence 3: shape (3072,), first 5 dims [-0.07893984 -0.00125954  0.10306348 -0.10213654  0.22425897]


# Borzoi Embeddings

In [ ]:
from embpy.models.dna_models import BorzoiWrapper
import torch


def random_dna(seq: int):  # noqa: D103
    return "".join(random.choice("ACTG") for _ in range(seq))


seq = random_dna(200_000)

wrapper = BorzoiWrapper("johahi/borzoi-replicate-0")
wrapper.load(device=torch.device("mps"))

embs = wrapper.embed(input=seq, pooling_strategy="mean")

print("Shape of embeddings :", embs.shape)
print("Value of embeddings (after pooling) :", embs)

Running Borzoi model …
tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]])
torch.Size([1, 1536, 6144])
Shape of embeddings : (1536,)
Value of embeddings (after pooling) : [-0.48280635 -0.20134836 -0.583343   ...  0.16279209 -1.2780379
 -0.4207624 ]


In [7]:
# In [1]: imports
import torch
import torch.nn.functional as F

from embpy.models.dna_models import BorzoiWrapper
from borzoi_pytorch import Borzoi


# 1) Utility to generate a fixed‐length DNA string
def random_dna_fixed(length: int) -> str:  # noqa: D103
    return "".join(random.choice("ACGT") for _ in range(length))


# 2) Instantiate & load your wrapper
wrapper = BorzoiWrapper("johahi/borzoi-replicate-0")
wrapper.load(device=torch.device("cpu"))

# 3) Generate a small batch of length-524288 sequences
batch_size = 3
seq_length = 524_288
dna_batch = [random_dna_fixed(seq_length) for _ in range(batch_size)]
print("All sequences length:", [len(s) for s in dna_batch])

# 4) Embed with your wrapper’s batch API
embs_wrapper = wrapper.embed_batch(dna_batch, pooling_strategy="mean")
print("\nWrapper.embed_batch results:")
for i, emb in enumerate(embs_wrapper):
    print(f"  Seq {i}: shape {emb.shape}")
    print(emb)

# 5) Cross-check via the raw Borzoi API (no pad/crop needed since L_in == SEQUENCE_LENGTH)
model = Borzoi.from_pretrained("johahi/borzoi-replicate-0").eval()


def direct_embed(seq: str) -> torch.Tensor:  # noqa: D103
    # map to indices & one-hot
    mapping = {"A": 0, "C": 1, "G": 2, "T": 3}
    idx = torch.tensor([[mapping[b] for b in seq]], dtype=torch.long)  # (1,524288)
    oh = F.one_hot(idx, num_classes=4).permute(0, 2, 1).float()  # (1,4,524288)

    # directly run get_embs_after_crop (no pad/crop needed)
    with torch.no_grad():
        embs = model.get_embs_after_crop(oh)  # (1, hidden_dim, bins)

    # mean‐pool over bins
    trunk = embs.squeeze(0)  # (hidden_dim, bins)
    return trunk.mean(dim=1)  # → (hidden_dim,)


print("\nDirect API results:")
for i, seq in enumerate(dna_batch):
    vec = direct_embed(seq).cpu().numpy()
    print(f"  Seq {i}: shape {vec.shape}")
    print(vec)

All sequences length: [524288, 524288, 524288]

Wrapper.embed_batch results:
  Seq 0: shape (1536,)
[-0.43352532 -0.13696681 -1.2183492  ...  0.26763576 -1.2105696
 -0.43681026]
  Seq 1: shape (1536,)
[-0.45959064 -0.08603456 -1.2036048  ...  0.29349053 -1.2051601
 -0.49734643]
  Seq 2: shape (1536,)
[-0.3804114  -0.15245233 -1.233302   ...  0.18367977 -1.1142911
 -0.49130872]

Direct API results:
  Seq 0: shape (1536,)
[-0.43352532 -0.13696681 -1.2183492  ...  0.26763576 -1.2105696
 -0.43681026]
  Seq 1: shape (1536,)
[-0.45959064 -0.08603456 -1.2036048  ...  0.29349053 -1.2051601
 -0.49734643]
  Seq 2: shape (1536,)
[-0.3804114  -0.15245233 -1.233302   ...  0.18367977 -1.1142911
 -0.49130872]


In [6]:
dna_batch

['...']

# Request Data from ENSEMBL to test the bed files and the FASTA sequences

In [3]:
from pyensembl import EnsemblRelease

# Load Ensembl release 109 (for GRCh38 / human)
data = EnsemblRelease(109)

# Download and index (only needed once)
data.download()
data.index()

# Fetch genes with name "TP53"
genes = data.genes_by_name("TP53")

# Access the first result
if genes:
    gene = genes[0]
    print("Gene:", gene.gene_name)
    print("Ensembl ID:", gene.gene_id)
    print("Chromosome:", gene.contig)
    print("Start:", gene.start)
    print("End:", gene.end)
    print("Strand:", gene.strand)
else:
    print("Gene not found.")

INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle


Gene: TP53
Ensembl ID: ENSG00000141510
Chromosome: 17
Start: 7661779
End: 7687538
Strand: -


In [31]:
import requests

gene = "BRCA1"
r = requests.get(
    f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene}?expand=1", headers={"Content-Type": "application/json"}
)
info = r.json()

chrom = info["seq_region_name"]
start = info["start"] - 1  # BED is 0-based
end = info["end"]
strand = "+" if info["strand"] == 1 else "-"
name = info["display_name"]

bed_line = f"{chrom}\t{start}\t{end}\t{name}\t{strand}\t{name}"
print(bed_line)

17	43044294	43170245	BRCA1	-	BRCA1


In [32]:
import requests

url = f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene}?expand=1"
headers = {"Content-Type": "application/json"}

response = requests.get(url, headers=headers)

if response.ok:
    data = response.json()
    print("Gene name:", data.get("display_name"))
    print("Description:", data.get("description"))
else:
    print(f"Failed to fetch info: {response.status_code}")

Gene name: BRCA1
Description: BRCA1 DNA repair associated [Source:HGNC Symbol;Acc:HGNC:1100]


In [29]:
# Construct API URL for sequence
fasta_url = f"https://rest.ensembl.org/sequence/region/human/{chrom}:{start + 1}..{end}:{strand}"

# Request FASTA
headers = {"Content-Type": "text/x-fasta"}
response = requests.get(fasta_url, headers=headers)

# Check response and print
if response.ok:
    print(response.text)
else:
    print(f"Failed to fetch FASTA: {response.status_code}")

>chromosome:GRCh38:17:43044295:43170245:-1
...
AAACACTTCCA



# How to get information from a molecule ?

In [36]:
import requests

# Define the compound name
compound_name = "aspirin"

# Fetch compound CID
cid_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{compound_name}/cids/JSON"
cid_response = requests.get(cid_url)
cid = cid_response.json()["IdentifierList"]["CID"][0]

# Fetch compound properties
properties = "MolecularFormula,MolecularWeight,CanonicalSMILES,InChIKey,IUPACName"
property_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/{properties}/JSON"
property_response = requests.get(property_url)
property_data = property_response.json()["PropertyTable"]["Properties"][0]

# Display the information
print(f"Compound: {compound_name}")
print(f"CID: {cid}")
print(f"IUPAC Name: {property_data['IUPACName']}")
print(f"Molecular Formula: {property_data['MolecularFormula']}")
print(f"Molecular Weight: {property_data['MolecularWeight']}")
print(f"SMILES: {property_data['CanonicalSMILES']}")
print(f"InChIKey: {property_data['InChIKey']}")

Compound: aspirin
CID: 2244
IUPAC Name: 2-acetyloxybenzoic acid
Molecular Formula: C9H8O4
Molecular Weight: 180.16
SMILES: CC(=O)OC1=CC=CC=C1C(=O)O
InChIKey: BSYNRYMUTXBXSQ-UHFFFAOYSA-N


In [38]:
import requests

molecule_name = "aspirin"
base_url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

# Fetch compound CID
cid_url = f"{base_url}/compound/name/{molecule_name}/cids/JSON"
cid_response = requests.get(cid_url)
if cid_response.ok:
    cid = cid_response.json()["IdentifierList"]["CID"][0]
    print(f"CID: {cid}")
else:
    print("Compound not found.")
    exit()

# Fetch compound properties
properties = "MolecularFormula,MolecularWeight,CanonicalSMILES,InChI,InChIKey,IUPACName"
prop_url = f"{base_url}/compound/cid/{cid}/property/{properties}/JSON"
prop_response = requests.get(prop_url)
if prop_response.ok:
    props = prop_response.json()["PropertyTable"]["Properties"][0]
    print("\nCompound Properties:")
    for key, value in props.items():
        print(f"{key}: {value}")
else:
    print("Failed to retrieve compound properties.")

# Fetch compound synonyms
synonyms_url = f"{base_url}/compound/cid/{cid}/synonyms/JSON"
syn_response = requests.get(synonyms_url)
if syn_response.ok:
    synonyms = syn_response.json()["InformationList"]["Information"][0]["Synonym"]
    print("\nSynonyms:")
    for synonym in synonyms[:10]:  # Display first 10 synonyms
        print(f"- {synonym}")
else:
    print("Failed to retrieve synonyms.")

CID: 2244

Compound Properties:
CID: 2244
MolecularFormula: C9H8O4
MolecularWeight: 180.16
CanonicalSMILES: CC(=O)OC1=CC=CC=C1C(=O)O
InChI: InChI=1S/C9H8O4/c1-6(10)13-8-5-3-2-4-7(8)9(11)12/h2-5H,1H3,(H,11,12)
InChIKey: BSYNRYMUTXBXSQ-UHFFFAOYSA-N
IUPACName: 2-acetyloxybenzoic acid

Synonyms:
- aspirin
- ACETYLSALICYLIC ACID
- 50-78-2
- 2-(Acetyloxy)benzoic acid
- Acetosal
- Acenterine
- Acetophen
- Acetosalin
- Aspirdrops
- Salcetogen


In [33]:
# import torch
# from evo2 import Evo2

# evo2_model = Evo2('evo2_7b')

# sequence = 'ACGT'
# input_ids = torch.tensor(
#     evo2_model.tokenizer.tokenize(sequence),
#     dtype=torch.int,
# ).unsqueeze(0).to('cuda:0')

# layer_name = 'blocks.28.mlp.l3'

# outputs, embeddings = evo2_model(input_ids, return_embeddings=True, layer_names=[layer_name])

# print('Embeddings shape: ', embeddings[layer_name].shape)

# ChemBERTa

In [9]:
import torch
from transformers import AutoTokenizer, AutoModel

# 2a) Specify the model name on Hugging Face Hub
MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"

# 2b) Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2c) Load the (masked‐LM / embedding) model
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()  # we’re doing inference only

RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(767, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-5): 6 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropou

In [10]:
# Example SMILES strings
smiles_list = ["CCO", "c1ccccc1"]

# 3a) Tokenize with padding/truncation to a uniform length
#     ChemBERTa’s tokenizer will split into character‐level tokens,
#     add special [CLS]/[SEP], etc.
encodings = tokenizer(smiles_list, padding=True, truncation=True, return_tensors="pt")
# encodings is a dict: { "input_ids": Tensor(batch_size×seq_len),
#                        "attention_mask": Tensor(batch_size×seq_len) }

In [11]:
# 4a) Move to GPU if available (optional)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
encodings = {k: v.to(device) for k, v in encodings.items()}

# 4b) Forward pass (no gradient needed)
with torch.no_grad():
    outputs = model(**encodings)
    # AutoModel returns a BaseModelOutput with:
    #   last_hidden_state shape = (batch_size, seq_len, hidden_dim)
    #   (there may be other fields—e.g. pooler_output—depending on the config)
    last_hidden = outputs.last_hidden_state

print("last_hidden_state.shape =", last_hidden.shape)

last_hidden_state.shape = torch.Size([2, 6, 768])


In [12]:
# Suppose cls_embedding = hidden at index 0 for each sequence
cls_embeddings = last_hidden[:, 0, :]  # shape: (batch_size, 768)

# Or mean‐pool across the seq_len dimension (ignoring padding via attention_mask)
attention_mask = encodings["attention_mask"].unsqueeze(-1)  # (batch_size, seq_len, 1)
masked_hidden = last_hidden * attention_mask  # zero‐out padded positions
sum_hidden = masked_hidden.sum(dim=1)  # sum over seq_len → (batch_size, 768)
lengths = attention_mask.sum(dim=1)  # how many real tokens (batch_size, 1)
mean_embeddings = sum_hidden / lengths  # (batch_size, 768)

print("CLS‐based embedding:", cls_embeddings.shape)
print("Mean‐pooled embedding:", mean_embeddings.shape)

CLS‐based embedding: torch.Size([2, 768])
Mean‐pooled embedding: torch.Size([2, 768])


In [13]:
import torch
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained("ibm/MoLFormer-XL-both-10pct", deterministic_eval=True, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("ibm/MoLFormer-XL-both-10pct", trust_remote_code=True)

smiles = ["Cn1c(=O)c2c(ncn2C)n(C)c1=O", "CC(=O)Oc1ccccc1C(=O)O"]
inputs = tokenizer(smiles, padding=True, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
outputs.pooler_output

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

configuration_molformer.py:   0%|          | 0.00/7.60k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ibm/MoLFormer-XL-both-10pct:
- configuration_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_molformer.py:   0%|          | 0.00/39.3k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ibm/MoLFormer-XL-both-10pct:
- modeling_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

A new version of the following files was downloaded from https://huggingface.co/ibm/MoLFormer-XL-both-10pct:
- tokenization_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/ibm/MoLFormer-XL-both-10pct:
- tokenization_molformer_fast.py
- tokenization_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


vocab.json:   0%|          | 0.00/41.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/54.0k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

tensor([[-0.4407,  0.3902,  0.7989,  ..., -0.6081, -0.0200,  0.0103],
        [ 0.5943,  0.4527,  0.3437,  ...,  0.1520, -0.3464,  0.5590]])

# Testing Out Embeddings

In [ ]:
import torch
import numpy as np

# 2) Import your BioEmbedder (which under the hood will use EnformerWrapper)
from embpy.embedder import BioEmbedder


# 1) Generate a random DNA string (you can choose any length; Enformer will pad or truncate to 196 608 bp)
def random_dna(length: int) -> str:  # noqa: D103
    return "".join(random.choices(["A", "C", "G", "T"], k=length))


# For example, pick length = 200 000 (slightly above 196 608 so Enformer will center‐crop):
dna_seq = random_dna(200_000)
print("Generated DNA length:", len(dna_seq))  # → 200000


# 3) Instantiate BioEmbedder and load the Enformer model onto CPU (or "cuda" if you have a GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedder = BioEmbedder(device=device)

# 4) Now call embed_gene with a dummy “gene identifier”—but since we already have the raw DNA,
#    we’ll bypass gene‐resolution and call the EnformerWrapper directly.
#    (Alternatively, if your GeneResolver can fetch a DNA string for an actual gene, you can use embed_gene.)

#   Here we manually get the Enformer wrapper from the embedder’s registry, then call embed(…).
model_name = "enformer_human_rough"
wrapper = embedder._get_model(model_name)  # this will load under the hood if not already loaded

# 5) Embed the random DNA sequence using Enformer
#    (EnformerWrapper will pad/truncate internally to exactly 196608, one-hot encode, run the model, pool)
embedding = wrapper.embed(input=dna_seq, pooling_strategy="mean")

print("Enformer embedding shape:", embedding.shape)
# Should be (3072,) because EnformerWrapper.TRUNK_OUTPUT_DIM = 3072

# 6) (Optional) Convert to float32 NumPy if needed:
embedding = embedding.astype(np.float32)

Generated DNA length: 200000


Enformer embedding shape: (3072,)


In [ ]:
import torch
import numpy as np
from embpy.embedder import BioEmbedder


# 1) Generate a random DNA string (you can choose any length; Enformer will pad or truncate to 196 608 bp)
def random_dna(length: int) -> str:  # noqa: D103
    return "".join(random.choices(["A", "C", "G", "T"], k=length))


# For example, pick length = 200 000 (slightly above 196 608 so Enformer will center‐crop):
dna_seq = random_dna(196_608)
print("Generated DNA length:", len(dna_seq))  # → 200000

# 2) Import your BioEmbedder (which under the hood will use EnformerWrapper)

# 3) Instantiate BioEmbedder and load the Enformer model onto CPU (or "cuda" if you have a GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedder = BioEmbedder(device=device)

# 4) Now call embed_gene with a dummy “gene identifier”—but since we already have the raw DNA,
#    we’ll bypass gene‐resolution and call the EnformerWrapper directly.
#    (Alternatively, if your GeneResolver can fetch a DNA string for an actual gene, you can use embed_gene.)

#   Here we manually get the Enformer wrapper from the embedder’s registry, then call embed(…).
model_name = "borzoi_v0"
wrapper = embedder._get_model(model_name)  # this will load under the hood if not already loaded

# 5) Embed the random DNA sequence using Enformer
#    (EnformerWrapper will pad/truncate internally to exactly 196608, one-hot encode, run the model, pool)
embedding = wrapper.embed(input=dna_seq, pooling_strategy="mean")

print("Borzoi embedding shape:", embedding.shape)
# Should be (3072,) because EnformerWrapper.TRUNK_OUTPUT_DIM = 3072

# 6) (Optional) Convert to float32 NumPy if needed:
embedding = embedding.astype(np.float32)

Generated DNA length: 196608
Running Borzoi model …
torch.Size([1, 1536, 16352])
Borzoi embedding shape: (1536,)


# ESM 2 embeddings 

In [12]:
# In [1]:
from embpy.embedder import BioEmbedder
import torch

# 1) Initialize your embedder on CPU (or "cuda"):
embedder = BioEmbedder(device=torch.device("cpu"))

# 2) See which models you have available:
print("Available models:", embedder.list_available_models())

# 3) Pick an ESM2 model key from that list:
model_key = "esm2_650M"

# 4) Generate a random protein (amino‐acid) sequence:
amino_acids = "ACDEFGHIKLMNPQRSTVWY"
seq_length = 200
random_protein = "".join(random.choices(amino_acids, k=seq_length))
print(f"Random protein ({seq_length} aa):", random_protein[:50] + "…")

# 5) Load the wrapper (this will download & cache the HF weights under-the-hood):
wrapper = embedder._get_model(model_key)

# 6) Embed your sequence (default is “mean” pooling; you can also use “max” or “cls”):
embedding = wrapper.embed(random_protein, pooling_strategy="mean")

# 7) Inspect the result:
print("Embedding shape:", embedding.shape)  # e.g. (768,) or however large your ESM-2 hidden dim is

Available models: ['bert_base_uncased', 'borzoi_v0', 'borzoi_v1', 'chemberta_zinc_v1', 'enformer_human_rough', 'esm2_150M', 'esm2_35M', 'esm2_3B', 'esm2_650M', 'esm2_8M', 'minilm_l6_v2', 'molformer_base']
Random protein (200 aa): NRMFPLCTWCYHGIVRYCSFTVPHPNCPKNDETRLTANEPEHPRQFWIGA…


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Embedding shape: (1280,)


# Chemberta

In [ ]:
import torch
import random
from transformers import AutoTokenizer, AutoModel

# 0) make everything reproducible
seed = 42
random.seed(seed)
torch.manual_seed(seed)

# 1) Pick a ChemBERTa checkpoint
model_name = "seyonec/ChemBERTa-zinc-base-v1"

# 2) Load tokenizer & model (and switch to eval)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).eval()  # <— IMPORTANT!


# 3) Tokenize your SMILES
def random_smiles(n: int) -> str:  # noqa: D103
    tokens = ["C", "N", "O"]
    return "".join(random.choices(tokens, k=n))


smiles = random_smiles(128)

inputs = tokenizer(smiles, padding=True, truncation=True, return_tensors="pt")

# 4) Forward pass (no gradient needed, dropout off because of eval())
with torch.no_grad():
    outputs = model(**inputs)

# 5) Extract the CLS embedding
cls_direct = outputs.last_hidden_state[:, 0, :].cpu()

# 6) Now do it through your wrapper
from embpy.embedder import BioEmbedder  # noqa: E402

embedder = BioEmbedder(device="cpu")
vec_cls = embedder.embed_molecule(smiles, model="chemberta_zinc_v1", pooling_strategy="cls")

# they should now be identical (up to floating‐point rounding)
print("Direct CLS:", cls_direct.shape, cls_direct[0, :5])
print("Wrapper CLS:", vec_cls.shape, vec_cls[:5])
# print("Max abs diff:", (cls_direct[0].numpy() - vec_cls).abs().max())

Direct CLS: torch.Size([1, 768]) tensor([ 0.0313, -0.8472, -0.9893,  0.2864, -0.7245])
Wrapper CLS: (768,) [ 0.03130991 -0.8472023  -0.98933     0.28640974 -0.72450495]


# MolFormer

In [1]:
from embpy.embedder import BioEmbedder
import torch

embedder = BioEmbedder(device=torch.device("cpu"))
# should list both chemberta_zinc_v1 and molformer_xl
print(embedder.list_available_models())

smiles = "CC(=O)Oc1ccccc1C(=O)O"
emb = embedder.embed_molecule(smiles, model="molformer_base", pooling_strategy="cls")
print("MolFormer embedding shape:", emb.shape)

['bert_base_uncased', 'borzoi_v0', 'borzoi_v1', 'chemberta_zinc_v1', 'enformer_human_rough', 'esm2_150M', 'esm2_35M', 'esm2_3B', 'esm2_650M', 'esm2_8M', 'minilm_l6_v2', 'molformer_base']


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


MolFormer embedding shape: (768,)


In [4]:
print(emb)

[ 5.94280064e-01  4.52650040e-01  3.43747914e-01  6.67825580e-01
 -6.99579656e-01  2.02832788e-01 -5.44053353e-02  8.19928646e-01
  2.11379856e-01 -1.15139507e-01 -6.97321951e-01  7.15679109e-01
  5.20055711e-01  2.29307845e-01 -4.24501389e-01  1.97534814e-01
  3.03925332e-02 -2.96359032e-01 -4.01238978e-01 -5.29726088e-01
 -7.38125205e-01  6.60815015e-02  3.13309371e-01  7.46078014e-01
  2.41676614e-01  4.75266218e-01 -1.43960690e+00 -2.91483134e-01
 -6.32730305e-01  7.32680500e-01 -6.61421657e-01 -6.26281261e-01
 -1.05253361e-01  3.25632334e-01  6.07707977e-01  4.24847901e-01
  3.78585994e-01 -3.01597148e-01 -6.80134296e-01 -1.79451853e-01
 -1.41719967e-01  4.31221336e-01 -1.47480192e-02 -1.45637870e-01
 -1.72145277e-01 -2.95462102e-01  1.42974734e-01 -7.01401532e-02
  2.54375458e-01  4.04036850e-01  6.48585916e-01  1.57276704e-03
  5.96743226e-01  7.34008610e-01  1.47559628e-01 -1.99199796e-01
 -3.74050140e-01  2.49511153e-01 -1.34057164e-01 -5.61859123e-02
  3.31139356e-01  4.65937

In [3]:
import torch
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained("ibm/MoLFormer-XL-both-10pct", deterministic_eval=True, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("ibm/MoLFormer-XL-both-10pct", trust_remote_code=True)

smiles = "CC(=O)Oc1ccccc1C(=O)O"
inputs = tokenizer(smiles, padding=True, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
outputs.pooler_output

tensor([[ 5.9428e-01,  4.5265e-01,  3.4375e-01,  6.6783e-01, -6.9958e-01,
          2.0283e-01, -5.4405e-02,  8.1993e-01,  2.1138e-01, -1.1514e-01,
         -6.9732e-01,  7.1568e-01,  5.2006e-01,  2.2931e-01, -4.2450e-01,
          1.9753e-01,  3.0393e-02, -2.9636e-01, -4.0124e-01, -5.2973e-01,
         -7.3813e-01,  6.6082e-02,  3.1331e-01,  7.4608e-01,  2.4168e-01,
          4.7527e-01, -1.4396e+00, -2.9148e-01, -6.3273e-01,  7.3268e-01,
         -6.6142e-01, -6.2628e-01, -1.0525e-01,  3.2563e-01,  6.0771e-01,
          4.2485e-01,  3.7859e-01, -3.0160e-01, -6.8013e-01, -1.7945e-01,
         -1.4172e-01,  4.3122e-01, -1.4748e-02, -1.4564e-01, -1.7215e-01,
         -2.9546e-01,  1.4297e-01, -7.0140e-02,  2.5438e-01,  4.0404e-01,
          6.4859e-01,  1.5728e-03,  5.9674e-01,  7.3401e-01,  1.4756e-01,
         -1.9920e-01, -3.7405e-01,  2.4951e-01, -1.3406e-01, -5.6186e-02,
          3.3114e-01,  4.6594e-02,  5.4272e-01, -9.9487e-02,  4.9468e-01,
         -2.7091e-02,  2.3985e-01,  1.

# Checking for this API wrappers, to see if they work 

In [1]:
from embpy.resources.gene_resolver import GeneResolver

# Initialize the resolver
resolver = GeneResolver()

In [3]:
resolver = GeneResolver()
seq = resolver.get_dna_sequence("BRCA1", id_type="symbol")
print(seq)

...


In [3]:
import requests

gene = "BRCA1"

# Step 1: Get gene coordinates and metadata
lookup_url = f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene}?expand=1"
headers = {"Content-Type": "application/json"}
r = requests.get(lookup_url, headers=headers)
r.raise_for_status()
info = r.json()

# Extract coordinates
chrom = info["seq_region_name"]
start = info["start"]
end = info["end"]
strand = info["strand"]
gene_id = info["id"]
name = info["display_name"]

# BED format (0-based start)
bed_line = f"{chrom}\t{start - 1}\t{end}\t{name}\t0\t{'+' if strand == 1 else '-'}"
print("BED line:")
print(bed_line)

# Step 2: Get the DNA sequence using Ensembl's sequence endpoint
sequence_url = f"https://rest.ensembl.org/sequence/id/{gene_id}?type=genomic"
seq_response = requests.get(sequence_url, headers={"Content-Type": "text/x-fasta"})
seq_response.raise_for_status()
fasta = seq_response.text

print("\nFASTA sequence:")
print(fasta)

BED line:
17	43044294	43170245	BRCA1	0	-

FASTA sequence:
>ENSG00000012048.26 chromosome:GRCh38:17:43044295:43170245:-1
...
AAACACTTCCA



# get_protein_sequence

In [6]:
from embpy.resources.gene_resolver import GeneResolver  # Replace with actual import
import time

# Initialize
resolver = GeneResolver()

# Example identifiers to test
genes_to_test = [
    ("TP53", "symbol"),
    ("ENSG00000139618", "ensembl_id"),  # BRCA2
    ("P38398", "uniprot_id"),  # BRCA1
]

# Benchmark get_protein_sequence
print("Testing get_protein_sequence:")
for identifier, id_type in genes_to_test:
    print(f"\nFetching protein for {id_type.upper()} '{identifier}':")
    start = time.time()
    seq = resolver.get_protein_sequence(identifier, id_type)
    end = time.time()
    if seq:
        print(f"✓ Length: {len(seq)} aa | Time: {end - start:.2f}s")
        print(f"Start: {seq}")
    else:
        print("✗ Sequence not found.")

ERROR:root:Error fetching protein sequence for TP53: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/search?query=gene_exact:TP53%20AND%20organism:%22human%22&format=json&fields=accession


Testing get_protein_sequence:

Fetching protein for SYMBOL 'TP53':
✗ Sequence not found.

Fetching protein for ENSEMBL_ID 'ENSG00000139618':


ERROR:root:Error fetching protein sequence for ENSG00000139618: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/search?query=dbxref:Ensembl:ENSG00000139618&format=json&fields=accession


✗ Sequence not found.

Fetching protein for UNIPROT_ID 'P38398':
✓ Length: 1863 aa | Time: 0.23s
Start: MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLKLLNQKKGPSQCPLCKNDITKRSLQESTRFSQLVEELLKIICAFQLDTGLEYANSYNFAKKENNSPEHLKDEVSIIQSMGYRNRAKRLLQSEPENPSLQETSLSVQLSNLGTVRTLRTKQRIQPQKTSVYIELGSDSSEDTVNKATYCSVGDQELLQITPQGTRDEISLDSAKKAACEFSETDVTNTEHHQPSNNDLNTTEKRAAERHPEKYQGSSVSNLHVEPCGTNTHASSLQHENSSLLLTKDRMNVEKAEFCNKSKQPGLARSQHNRWAGSKETCNDRRTPSTEKKVDLNADPLCERKEWNKQKLPCSENPRDTEDVPWITLNSSIQKVNEWFSRSDELLGSDDSHDGESESNAKVADVLDVLNEVDEYSGSSEKIDLLASDPHEALICKSERVHSKSVESNIEDKIFGKTYRKKASLPNLSHVTENLIIGAFVTEPQIIQERPLTNKLKRKRRPTSGLHPEDFIKKADLAVQKTPEMINQGTNQTEQNGQVMNITNSGHENKTKGDSIQNEKNPNPIESLEKESAFKTKAEPISSSISNMELELNIHNSKAPKKNRLRRKSSTRHIHALELVVSRNLSPPNCTELQIDSCSSSEEIKKKKYNQMPVRHSRNLQLMEGKEPATGAKKSNKPNEQTSKRHDSDTFPELKLTNAPGSFTKCSNTSELKEFVNPSLPREEKEEKLETVKVSNNAEDPKDLMLSGERVLQTERSVESSSISLVPGTDYGTQESISLLEVSTLGKAKTEPNKCVSQCAAFENPKGLIHGCSKDNRNDTEGFKYPLGHEVNHSRETSIEMEESELDAQYLQNTFKVSKRQSFAPFSNPGNAEEECATFSAHSGSLKKQS

In [3]:
from embpy.resources.gene_resolver import GeneResolver
import time

resolver = GeneResolver()

genes_to_test = [
    ("TP53", "symbol"),
    ("P04637", "uniprot_id"),
    ("ENSP00000419060", "ensembl_id"),  # <-- transcript ID, not gene ID!
]

print("Testing get_protein_sequence:\n")
for identifier, id_type in genes_to_test:
    print(f"Fetching protein for {id_type.upper()} '{identifier}':")
    start = time.time()
    sequence = resolver.get_protein_sequence(identifier=identifier, id_type=id_type)
    elapsed = time.time() - start

    if sequence:
        print(f"✓ Success | Length: {len(sequence)} aa | Time: {elapsed:.2f}s")
        print(f"First 60 aa: {sequence[:60]}...\n")
    else:
        print("✗ Sequence not found.\n")

Testing get_protein_sequence:

Fetching protein for SYMBOL 'TP53':
✓ Success | Length: 393 aa | Time: 0.32s
First 60 aa: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...

Fetching protein for UNIPROT_ID 'P04637':
✓ Success | Length: 393 aa | Time: 0.12s
First 60 aa: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...

Fetching protein for ENSEMBL_ID 'ENSP00000419060':


ERROR:root:Error fetching protein sequence for ENSP00000419060: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/search?query=xref%3AEnsembl%3AENSP00000419060%20AND%20reviewed%3Atrue&format=json


✗ Sequence not found.



In [1]:
from embpy.resources.gene_resolver import GeneResolver

resolver = GeneResolver()

# Example: Get UniProt ID from Ensembl ID
ensembl_id = "ENSG00000141510"  # TP53
uniprot_id = resolver.get_uniprot_id_from_gene(ensembl_id, id_type="ensembl_id")

if uniprot_id:
    print(f"Resolved UniProt ID: {uniprot_id}")
    protein_seq = resolver.get_protein_sequence(uniprot_id, id_type="uniprot_id")
    print(f"Protein sequence length: {len(protein_seq)}\nFirst 60 aa: {protein_seq[:60]}...")
else:
    print("Could not resolve UniProt ID.")

Resolved UniProt ID: P04637
Protein sequence length: 393
First 60 aa: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...


In [1]:
from embpy.resources.gene_resolver import GeneResolver
import time

# Initialize resolver
resolver = GeneResolver()

# Example test cases: symbol, Ensembl gene ID, and UniProt accession
genes_to_test = [
    ("TP53", "symbol"),  # Human tumor suppressor gene
    ("ENSG00000141510", "ensembl_id"),  # Ensembl gene ID for TP53
    ("P04637", "uniprot_id"),  # UniProt accession for TP53 protein
]

# Run benchmark
print("🧬 Testing get_protein_sequence:\n")
for identifier, id_type in genes_to_test:
    print(f"🔍 Fetching protein for {id_type.upper()} '{identifier}':")
    start = time.time()
    sequence = resolver.get_protein_sequence(identifier=identifier, id_type=id_type)
    elapsed = time.time() - start

    if sequence:
        print(f"✅ Success | Length: {len(sequence)} aa | Time: {elapsed:.2f}s")
        print(f"🔢 Start of sequence: {sequence[:60]}...\n")
    else:
        print("❌ Failed to retrieve protein sequence.\n")

🧬 Testing get_protein_sequence:

🔍 Fetching protein for SYMBOL 'TP53':
✅ Success | Length: 393 aa | Time: 0.99s
🔢 Start of sequence: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...

🔍 Fetching protein for ENSEMBL_ID 'ENSG00000141510':
✅ Success | Length: 393 aa | Time: 0.89s
🔢 Start of sequence: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...

🔍 Fetching protein for UNIPROT_ID 'P04637':
✅ Success | Length: 393 aa | Time: 0.12s
🔢 Start of sequence: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...



In [1]:
from embpy.resources.gene_resolver import GeneResolver  # Update to actual import if needed

# Initialize the resolver
resolver = GeneResolver()

# Example gene identifiers
genes_to_test = [
    ("BRCA1", "symbol"),
    ("ENSG00000141510", "ensembl_id"),  # TP53 Ensembl ID
    ("P04637", "uniprot_id"),  # TP53 UniProt ID
]

# Format string (optional – can include more keys if MyGene.info provides them)
format_str = "Gene: {symbol}. Name: {name}. Summary: {summary}"

print("🧬 Testing get_gene_description:\n")
for identifier, id_type in genes_to_test:
    print(f"🔍 {id_type.upper()} '{identifier}':")
    description = resolver.get_gene_description(identifier, id_type=id_type, format_string=format_str)
    if description:
        print(f"✅ Description:\n{description}\n")
    else:
        print("❌ Description not found.\n")

🧬 Testing get_gene_description:

🔍 SYMBOL 'BRCA1':
✅ Description:
Gene: BRCA1. Name: BRCA1 DNA repair associated. Summary: This gene encodes a 190 kD nuclear phosphoprotein that plays a role in maintaining genomic stability, and it also acts as a tumor suppressor. The BRCA1 gene contains 22 exons spanning about 110 kb of DNA. The encoded protein combines with other tumor suppressors, DNA damage sensors, and signal transducers to form a large multi-subunit protein complex known as the BRCA1-associated genome surveillance complex (BASC). This gene product associates with RNA polymerase II, and through the C-terminal domain, also interacts with histone deacetylase complexes. This protein thus plays a role in transcription, DNA repair of double-stranded breaks, and recombination. Mutations in this gene are responsible for approximately 40% of inherited breast cancers and more than 80% of inherited breast and ovarian cancers. Alternative splicing plays a role in modulating the subcellular l

# Benchmarking gene_embeddings.py

In [ ]:
# %% [markdown]
# # Benchmark Enformer Embedding of a Single Gene/Transcript
#
# This will:
# 1. Instantiate the processor (either full-gene or TSS-only by which ID you pass).
# 2. load_data() → reads mart_export‐5.txt + loads the right FASTA.
# 3. load_model() → downloads + moves Enformer onto your GPU/CPU.
# 4. embed_row() → slices, pads, infers, pools.
# 5. Prints timing and output shapes.

# %%
import time
import numpy as np
from embpy.gene_embeddings import GeneEmbeddingProcessor  # adjust to the actual import path

# %% [markdown]
# ## 1) Full‐gene example
# Pass a Gene stable ID or HGNC symbol to test the full‐gene path.

# %%
proc_full = GeneEmbeddingProcessor(
    mart_file="/Users/grpinto/Documents/embpy/data/mart_export-5.txt",
    chromosome_folder="/Users/grpinto/Documents/embpy/data/chromosomes",
    gene_id="ENSG00000292327",  # full‐gene
    transcript_id=None,
)
proc_full.load_data()
proc_full.load_model()

row = proc_full.row
print("Embedding full gene:", row["HGNC symbol"], "on chr", row["Chromosome/scaffold name"])

t0 = time.time()
max1, mean1, med1, sym1 = proc_full.embed_row(row)
dt1 = time.time() - t0

print(f">>> Done in {dt1:.2f}s")
print("  max  vector length:", max1.shape)
print("  mean vector length:", mean1.shape)
print("  median length  :", med1.shape)

/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:81: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)


Embedding full gene: PPP2R3B on chr Y
>>> Done in 2.40s
  max  vector length: (3072,)
  mean vector length: (3072,)
  median length  : (3072,)


In [2]:
# %% [markdown]
# ## 2) TSS‐only example
# Pass a Transcript ID (or version) to test the TSS‐only path.

import time
import numpy as np
from embpy.gene_embeddings import GeneEmbeddingProcessor  # adjust to the actual import path

# # %%
proc_tss = GeneEmbeddingProcessor(
    mart_file="/Users/grpinto/Documents/embpy/data/mart_export-5.txt",
    chromosome_folder="/Users/grpinto/Documents/embpy/data/chromosomes",
    gene_id=None,
    transcript_id="ENST00000711115",  # TSS-only
)
proc_tss.load_data()
proc_tss.load_model()

row2 = proc_tss.row
print("Embedding TSS‐only for transcript", row2["Transcript stable ID"], "of gene", row2["HGNC symbol"])

t1 = time.time()
max2, mean2, med2, sym2 = proc_tss.embed_row(row2)
dt2 = time.time() - t1

print(f">>> Done in {dt2:.2f}s")
print("  max  vector length:", max2.shape)
print("  mean vector length:", mean2.shape)
print("  median length  :", med2.shape)

/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:81: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)


Embedding TSS‐only for transcript ENST00000711115 of gene PPP2R3B
2560
GAGCAGTGGTGCAATCTCAGCTCACAGTAATCTCCGACTCCCAGCTTCAAGCATTTCTCCTGCCTCAGCCTCCTGAGTAGCTGAGACTTCAGGTGCGCCACCACGCCTGGCTAATTTGTGTATTTTTAGCAGAGATGGGGTTTCGCCATGTTGCCCAGGCTGGTCTCGAACTCCTGAACTCAAGTGATCTGCCTGCCTCCAACTCCCAAATGCTGGAATCACAGGCGTGAGTCACCATGCCTGACCAATATAAACAATATTAAAACACCTGCAGCTTCCTGTCAGATCCTGATGAACAGACCCCTCTGTTCCACCAGCCGTAACTACAGCTTTGACTGGGAAAAGACTGATTCCAGGCCAGGTGTGGTGGCTCACGCCTGTCATCCCAGCACTTTGAGAGGCTGAGGCGGGTGGATCACCTGAGGTCAGGAGTTTGAGACCAGTCTAGCCAACAAGGTGAAAGCCTGCCTCTACTAAAAATACAAAAATTAGCCGGGGGTGGTGGTGTGCGCCTGTAATCACAGCTACTCGGGAGGCTGAGGGAGGAGAATCGCTTGAACCCGGGAGGCGGAGGTTGCAGTGAGCTGAGATTGCACCATTGCATCTCAGCCTGGGTGAGAGTGAGAATCTGTCTCAAAAAAAAAAAAAAAAAAAAAAAACTGATTGCAATCACTTTATCCTGGTAACTACTCACCACGGACTGGTTCTGGCCGGTTGACAGAGGCTGCAGAGTTGCTTCACCTTTTGACCTAGGGGGCCTAACCATAATGCATTTAAATGTTAAGTCTCCGCTCCAAGGTGAACTCGGGAGTAGGTAACATGCATGTTTGTTCAATACCCATGCGTCAGGACACCCTTGGTGAATATCCATAGCTCTTCCTATAACTTCTTGAATATATACACTTGGCCAACCCACTCAGCATAAATTC

In [3]:
import time
import numpy as np
from embpy.gene_embeddings import GeneEmbeddingProcessor  # adjust import to wherever you put it

# 1) point to your files
MART_FILE = "/Users/grpinto/Documents/embpy/data/mart_export-5.txt"
CHROM_FOLDER = "/Users/grpinto/Documents/embpy/data/chromosomes"

# 2) some example IDs (first two genes and first two transcripts from your mart file)
gene_ids = ["ENSG00000292327", "ENSG00000292328"]
tx_ids = ["ENST00000711115", "ENST00000711114"]

# 3) make one processor instance (we'll re-use it)
proc = GeneEmbeddingProcessor(mart_file=MART_FILE, chromosome_folder=CHROM_FOLDER, gene_id=None, transcript_id=None)

# 4) warm up once for genes
print("→ Benchmarking  gene embeddings …")
t0 = time.time()
out_genes = proc.process_batch(gene_ids, id_type="gene")
t1 = time.time()
print(f"  • genes: {len(gene_ids)} → max-array {out_genes['max'].shape}, ∅time {t1 - t0:.2f}s")

# 5) now for transcripts
print("→ Benchmarking  TSS embeddings …")
t0 = time.time()
out_tss = proc.process_batch(tx_ids, id_type="transcript")
t1 = time.time()
print(f"  • tx   : {len(tx_ids)} → max-array {out_tss['max'].shape}, ∅time {t1 - t0:.2f}s")

# 6) peek at a few values
print("\nExample pooled vectors (first gene):")
print("  max[0,:5]   =", out_genes["max"][0, :5])
print("  mean[0,:5]  =", out_genes["mean"][0, :5])
print("  median[0,:5]=", out_genes["median"][0, :5])

→ Benchmarking  gene embeddings …


Batch embedding:   0%|          | 0/2 [00:00<?, ?it/s]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:81: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)
Batch embedding:  50%|█████     | 1/2 [00:03<00:03,  3.99s/it]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:81: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)
Batch embedding: 100%|██████████| 2/2 [00:07<00:00,  3.92s/it]


  • genes: 2 → max-array (2, 3072), ∅time 7.84s
→ Benchmarking  TSS embeddings …


Batch embedding:   0%|          | 0/2 [00:00<?, ?it/s]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:81: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)
Batch embedding:  50%|█████     | 1/2 [00:03<00:03,  3.82s/it]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:81: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)
Batch embedding: 100%|██████████| 2/2 [00:07<00:00,  3.85s/it]

  • tx   : 2 → max-array (2, 3072), ∅time 7.69s

Example pooled vectors (first gene):
  max[0,:5]   = [-0.05213138  1.6608177   1.0175893   0.3102237   0.38942063]
  mean[0,:5]  = [-0.14548557  0.00345589  0.32418683 -0.10733108 -0.00293958]
  median[0,:5]= [-1.50084645e-01 -3.79400699e-05  3.32294464e-01 -1.17599234e-01
 -3.52658816e-02]


In [ ]:
import time
from embpy.gene_embeddings import GeneEmbeddingProcessor

# 1) Specify just one gene
genes = ["PKIB"]
print(f"Testing embedding on gene list: {genes}")

# 2) Instantiate your processor
MART_FILE = "/Users/grpinto/Documents/embpy/data/mart_export-5.txt"
CHROMOSOME_DIR = "/Users/grpinto/Documents/embpy/data/chromosomes"

proc = GeneEmbeddingProcessor(
    mart_file=MART_FILE,
    chromosome_folder=CHROMOSOME_DIR,
)

# 3) Run the batch
t0 = time.time()
out = proc.process_batch(genes, id_type="gene")
t1 = time.time()
print(f"\n→ Finished embedding {len(out['genes'])} gene(s) in {t1 - t0:.1f}s\n")

# 4) Inspect shapes
print("max shape   :", out["max"].shape)
print("mean shape  :", out["mean"].shape)
print("median shape:", out["median"].shape)

# 5) Peek at the results for PKIB
print("\nGene symbol      :", out["genes"][0])
print("First 5 dims of max:", out["max"][0, :5])
print("First 5 dims of mean:", out["mean"][0, :5])
print("First 5 dims of median:", out["median"][0, :5])

Testing embedding on gene list: ['PKIB']


Batch embedding:   0%|          | 0/1 [00:00<?, ?it/s]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:81: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)


mps


Batch embedding: 100%|██████████| 1/1 [00:04<00:00,  4.90s/it]


→ Finished embedding 1 gene(s) in 4.9s

{'max': array([[ 0.3837057 , -0.        ,  0.38831705, ..., -0.        ,
        -0.        ,  0.7710912 ]], dtype=float32), 'mean': array([[-0.13448213, -0.00119984, -0.13498358, ..., -0.00033979,
        -0.10879367, -0.06621195]], dtype=float32), 'median': array([[-1.5187401e-01, -9.3944065e-05, -1.5622017e-01, ...,
        -2.6351702e-04, -1.1154340e-01, -1.3446763e-01]], dtype=float32), 'genes': array(['PKIB'], dtype='<U4')}
max shape   : (1, 3072)
mean shape  : (1, 3072)
median shape: (1, 3072)

Gene symbol      : PKIB
First 5 dims of max: [ 0.3837057  -0.          0.38831705 -0.          0.65603   ]
First 5 dims of mean: [-0.13448213 -0.00119984 -0.13498358 -0.11590254  0.03347152]
First 5 dims of median: [-1.5187401e-01 -9.3944065e-05 -1.5622017e-01 -1.1780445e-01
  2.4501063e-02]


In [ ]:
# 1) Imports & load your CCLE table
import pandas as pd
import time

from embpy.gene_embeddings import GeneEmbeddingProcessor

# path to your files
CCLE_CSV = "/Users/grpinto/Documents/embpy/data/ccle_300.csv"
MART_FILE = "/Users/grpinto/Documents/embpy/data/mart_export-5.txt"
CHROMOSOME_DIR = "/Users/grpinto/Documents/embpy/data/chromosomes"

# read CCLE, assume gene symbols are in the first (unnamed) column
ccle = pd.read_csv(CCLE_CSV, index_col=0)
genes = list(ccle.index)
print(f"This were the genes found {genes}")
print(f"Found {len(genes)} genes in CCLE file.")

# 2) Instantiate your processor (no need to set gene_id or tx_id here)
proc = GeneEmbeddingProcessor(
    mart_file=MART_FILE,
    chromosome_folder=CHROMOSOME_DIR,
)

# 3) Run the batch
t0 = time.time()
out = proc.process_batch(genes, id_type="gene")
t1 = time.time()
print(f"\n→ Finished embedding {len(out['genes'])} genes in {t1 - t0:.1f}s\n")

# 4) Inspect shapes
print("max shape   :", out["max"].shape)
print("mean shape  :", out["mean"].shape)
print("median shape:", out["median"].shape)

# 5) Peek at the first few entries
print("\nFirst 5 gene symbols:", out["genes"][:5])
print("First 5 dims of max[0]:", out["max"][0, :5])
print("First 5 dims of mean[0]:", out["mean"][0, :5])
print("First 5 dims of median[0]:", out["median"][0, :5])

This were the genes found ['PKIB', 'PKIG', 'PKLR', 'PKM', 'PKMYT1', 'PKN1', 'PKN2', 'PKN3', 'PKNOX1', 'PKNOX2', 'PKP1', 'PKP2', 'PKP3', 'PKP4', 'PLA1A', 'PLA2G10', 'PLA2G12A', 'PLA2G12B', 'PLA2G15', 'PLA2G1B', 'PLA2G2A', 'PLA2G2C', 'PLA2G2D', 'PLA2G2E', 'PLA2G2F', 'PLA2G3', 'PLA2G4A', 'PLA2G4B', 'PLA2G4C', 'PLA2G4D', 'PLA2G4E', 'PLA2G4F', 'PLA2G5', 'PLA2G6', 'PLA2G7', 'PLA2R1', 'PLAA', 'PLAAT1', 'PLAAT2', 'PLAAT3', 'PLAAT4', 'PLAAT5', 'PLAC1', 'PLAC8', 'PLAC8L1', 'PLAC9', 'PLAG1', 'PLAGL1', 'PLAGL2', 'PLAT', 'PLAU', 'PLAUR', 'PLB1', 'PLBD1', 'PLBD2', 'PLCB1', 'PLCB2', 'PLCB3', 'PLCB4', 'PLCD1', 'PLCD3', 'PLCD4', 'PLCE1', 'PLCG1', 'PLCG2', 'PLCH1', 'PLCH2', 'PLCL1', 'PLCL2', 'PLCXD1', 'PLCXD2', 'PLCXD3', 'PLCZ1', 'PLD1', 'PLD2', 'PLD3', 'PLD4', 'PLD5', 'PLD6', 'PLEC', 'PLEK', 'PLEK2', 'PLEKHA1', 'PLEKHA2', 'PLEKHA3', 'PLEKHA4', 'PLEKHA5', 'PLEKHA6', 'PLEKHA7', 'PLEKHA8', 'PLEKHB1', 'PLEKHB2', 'PLEKHD1', 'PLEKHF1', 'PLEKHF2', 'PLEKHG1', 'PLEKHG2', 'PLEKHG3', 'PLEKHG4', 'PLEKHG4B', 'PLEKH

Batch embedding:   0%|          | 0/18978 [00:00<?, ?it/s]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:81: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)
Batch embedding:   0%|          | 1/18978 [00:04<22:10:57,  4.21s/it]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:81: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)
Batch embedding:   0%|          | 2/18978 [00:07<20:43:46,  3.93s/it]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:81: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)
Batch embedding:   0%|          | 3/18978 [00:12<21:36:52,  4.10s/it]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:81: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import

KeyboardInterrupt: 

In [1]:
# %% [markdown]
# ## Single‐Gene Embedding Test: Enformer vs. Borzoi

# %%
import time
from embpy.gene_embeddings import GeneEmbeddingProcessor

# 1) Specify just one gene
genes = ["PKIB"]
print(f"Testing embedding on gene list: {genes}")

# 2) File paths
MART_FILE = "/Users/grpinto/Documents/embpy/data/mart_export-5.txt"
CHROMOSOME_DIR = "/Users/grpinto/Documents/embpy/data/chromosomes"

# %%
# 3) Loop over both models
for model_type in ["enformer", "borzoi"]:
    print(f"\n--- Running {model_type.upper()} on {genes[0]} ---")
    # instantiate with the desired model
    proc = GeneEmbeddingProcessor(
        mart_file=MART_FILE,
        chromosome_folder=CHROMOSOME_DIR,
        model_type=model_type,
    )

    # run batch
    t0 = time.time()
    out = proc.process_batch(genes, id_type="gene")
    dt = time.time() - t0

    print(f"→ Finished embedding {len(out['genes'])} gene(s) in {dt:.2f}s")
    print("  max shape   :", out["max"].shape)
    print("  mean shape  :", out["mean"].shape)
    print("  median shape:", out["median"].shape)

    # peek at the first few dimensions
    print(f"\nResults for {out['genes'][0]} ({model_type}):")
    print("  max   [:5] →", out["max"][0, :5])
    print("  mean  [:5] →", out["mean"][0, :5])
    print("  median[:5] →", out["median"][0, :5])

Testing embedding on gene list: ['PKIB']

--- Running ENFORMER on PKIB ---


Batch embedding:   0%|          | 0/1 [00:00<?, ?it/s]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:80: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)
Batch embedding: 100%|██████████| 1/1 [00:05<00:00,  5.12s/it]


→ Finished embedding 1 gene(s) in 5.16s
  max shape   : (1, 896)
  mean shape  : (1, 896)
  median shape: (1, 896)

Results for PKIB (enformer):
  max   [:5] → [4.577341  3.9423394 3.8128498 3.350918  3.25428  ]
  mean  [:5] → [-0.00069916 -0.00629943 -0.00540117 -0.01105045 -0.02002696]
  median[:5] → [-0.0177918  -0.01853262 -0.01919453 -0.02146341 -0.02277789]

--- Running BORZOI on PKIB ---


Batch embedding:   0%|          | 0/1 [00:00<?, ?it/s]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:80: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)
Batch embedding: 100%|██████████| 1/1 [00:06<00:00,  6.20s/it]

→ Finished embedding 1 gene(s) in 6.21s
  max shape   : (1, 1536)
  mean shape  : (1, 1536)
  median shape: (1, 1536)

Results for PKIB (borzoi):
  max   [:5] → [ 2.3643425  1.7859746 20.8771    52.84368    3.149362 ]
  mean  [:5] → [ 0.09549584  0.15841796 -0.57864946 -0.14974453  1.2208028 ]
  median[:5] → [ 0.11217407  0.4668538  -0.4637744  -0.44559485  1.828117  ]


In [1]:
# %% [markdown]
# ## Single‐Gene Embedding & Direct Borzoi API Comparison

# %%
import time
import torch
import numpy as np

from embpy.gene_embeddings import GeneEmbeddingProcessor
from borzoi_pytorch import Borzoi

# 1) Specify just one gene
genes = ["PKIB"]
print(f"Testing embedding on gene list: {genes}")

# 2) File paths
MART_FILE = "/Users/grpinto/Documents/embpy/data/mart_export-5.txt"
CHROMOSOME_DIR = "/Users/grpinto/Documents/embpy/data/chromosomes"

# %%
# 3) Run through Enformer & Borzoi via GeneEmbeddingProcessor
results = {}
for model_type in ["enformer", "borzoi"]:
    print(f"\n--- Running {model_type.upper()} on {genes[0]} ---")
    proc = GeneEmbeddingProcessor(
        mart_file=MART_FILE,
        chromosome_folder=CHROMOSOME_DIR,
        model_type=model_type,
    )
    t0 = time.time()
    out = proc.process_batch(genes, id_type="gene")
    dt = time.time() - t0

    print(f"→ Finished embedding {len(out['genes'])} gene(s) in {dt:.2f}s")
    print("  max shape   :", out["max"].shape)
    print("  mean shape  :", out["mean"].shape)
    print("  median shape:", out["median"].shape)
    print(f"  mean[:5]    → {np.round(out['mean'][0, :5], 4)}")

    results[model_type] = (proc, out)

# %%
# 4) Extract PKIB sequence from processor & compare direct Borzoi
proc_bor, out_bor = results["borzoi"]
proc_bor.load_data()
row = proc_bor.row

# slice the DNA exactly as embed_row does, using the bin-size helper
if proc_bor.region == "full_gene":
    start = int(row[proc_bor.start_col])
    end = int(row[proc_bor.end_col])
    seq = proc_bor.chrom_seq[start:end]
else:
    seq = proc_bor._get_tss_window(row)  # now uses 32 bp bins × N_TSS_BINS

# 5) Preprocess + direct Borzoi API
oh = proc_bor._preprocess_borzoi(seq)  # (1,4,524288) tensor
model = Borzoi.from_pretrained("johahi/borzoi-replicate-0").to(proc_bor.device).eval()
with torch.no_grad():
    embs = model.get_embs_after_crop(oh)  # (1, hidden_dim, bins)
trunk = embs.squeeze(0)  # (hidden_dim, bins)
direct_mean = trunk.mean(dim=1).cpu().numpy()

# 6) Compare wrapper vs. direct
wrapper_mean = out_bor["mean"][0]
print("\n--- Direct Borzoi API vs. Processor’s mean embedding ---")
print("Direct mean[:5]   →", np.round(direct_mean[:5], 4))
print("Wrapper mean[:5]  →", np.round(wrapper_mean[:5], 4))
print("Max abs diff      →", np.round(np.max(np.abs(direct_mean - wrapper_mean)), 6))

Testing embedding on gene list: ['PKIB']

--- Running ENFORMER on PKIB ---


Batch embedding:   0%|          | 0/1 [00:00<?, ?it/s]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:87: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)
Batch embedding: 100%|██████████| 1/1 [00:04<00:00,  4.49s/it]


→ Finished embedding 1 gene(s) in 4.50s
  max shape   : (1, 896)
  mean shape  : (1, 896)
  median shape: (1, 896)
  mean[:5]    → [-0.0007 -0.0063 -0.0054 -0.0111 -0.02  ]

--- Running BORZOI on PKIB ---


Batch embedding:   0%|          | 0/1 [00:00<?, ?it/s]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:87: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)
Batch embedding: 100%|██████████| 1/1 [00:04<00:00,  4.96s/it]


→ Finished embedding 1 gene(s) in 4.96s
  max shape   : (1, 1536)
  mean shape  : (1, 1536)
  median shape: (1, 1536)
  mean[:5]    → [ 0.0955  0.1584 -0.5786 -0.1497  1.2208]


/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:87: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)



--- Direct Borzoi API vs. Processor’s mean embedding ---
Direct mean[:5]   → [ 0.0955  0.1584 -0.5786 -0.1497  1.2208]
Wrapper mean[:5]  → [ 0.0955  0.1584 -0.5786 -0.1497  1.2208]
Max abs diff      → 0.0


In [1]:
# %% [markdown]
# ## Transcript Embedding Test: Enformer vs. Borzoi on ENST00000711113

# %%
import time
import torch
import numpy as np

from embpy.gene_embeddings import GeneEmbeddingProcessor
from borzoi_pytorch import Borzoi

# 1) Specify just one transcript
tx_ids = ["ENST00000711113"]
print(f"Testing embedding on transcript list: {tx_ids}")

# 2) File paths
MART_FILE = "/Users/grpinto/Documents/embpy/data/mart_export-5.txt"
CHROMOSOME_DIR = "/Users/grpinto/Documents/embpy/data/chromosomes"

# %%
# 3) Run through Enformer & Borzoi via GeneEmbeddingProcessor in transcript mode
results = {}
for model_type in ["enformer", "borzoi"]:
    print(f"\n--- Running {model_type.upper()} on {tx_ids[0]} ---")
    proc = GeneEmbeddingProcessor(
        mart_file=MART_FILE,
        chromosome_folder=CHROMOSOME_DIR,
        model_type=model_type,
        gene_id=None,
        transcript_id=tx_ids[0],
    )
    t0 = time.time()
    out = proc.process_batch(tx_ids, id_type="transcript")
    dt = time.time() - t0

    print(f"→ Finished embedding {len(out['genes'])} transcript(s) in {dt:.2f}s")
    print("  max shape   :", out["max"].shape)
    print("  mean shape  :", out["mean"].shape)
    print("  median shape:", out["median"].shape)
    print(f"  mean[:5]    → {np.round(out['mean'][0, :5], 4)}")
    results[model_type] = (proc, out)

# %%
# 4) Cross‐check direct Borzoi API on the same TSS‐only slice
proc_bor, out_bor = results["borzoi"]
proc_bor.load_data()
row = proc_bor.row

# use the same _get_tss_window for TSS-only
seq = proc_bor._get_tss_window(row)

# preprocess and run Borzoi directly
oh = proc_bor._preprocess_borzoi(seq)
model = Borzoi.from_pretrained("johahi/borzoi-replicate-0").to(proc_bor.device).eval()
with torch.no_grad():
    embs = model.get_embs_after_crop(oh)
trunk = embs.squeeze(0)
direct_mean = trunk.mean(dim=1).cpu().numpy()

# 5) Compare wrapper vs. direct
wrapper_mean = out_bor["mean"][0]
print("\n--- Direct Borzoi API vs. Processor’s mean embedding ---")
print("Direct mean[:5]  →", np.round(direct_mean[:5], 4))
print("Wrapper mean[:5] →", np.round(wrapper_mean[:5], 4))
print("Max abs diff     →", np.round(np.max(np.abs(direct_mean - wrapper_mean)), 6))

Testing embedding on transcript list: ['ENST00000711113']

--- Running ENFORMER on ENST00000711113 ---


Batch embedding:   0%|          | 0/1 [00:00<?, ?it/s]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:87: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)
Batch embedding: 100%|██████████| 1/1 [00:04<00:00,  4.14s/it]


→ Finished embedding 1 transcript(s) in 4.15s
  max shape   : (1, 896)
  mean shape  : (1, 896)
  median shape: (1, 896)
  mean[:5]    → [-0.0046 -0.0046 -0.0046 -0.0046 -0.0047]

--- Running BORZOI on ENST00000711113 ---


Batch embedding:   0%|          | 0/1 [00:00<?, ?it/s]/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:87: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)
Batch embedding: 100%|██████████| 1/1 [00:05<00:00,  5.18s/it]


→ Finished embedding 1 transcript(s) in 5.18s
  max shape   : (1, 1536)
  mean shape  : (1, 1536)
  median shape: (1, 1536)
  mean[:5]    → [-0.0213 -0.1239 -0.4601 -0.652   1.2242]


/Users/grpinto/Documents/embpy/src/embpy/gene_embeddings.py:87: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)



--- Direct Borzoi API vs. Processor’s mean embedding ---
Direct mean[:5]  → [-0.0213 -0.1239 -0.4601 -0.652   1.2242]
Wrapper mean[:5] → [-0.0213 -0.1239 -0.4601 -0.652   1.2242]
Max abs diff     → 0.0


In [1]:
import torch
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

# pick device automatically
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # for Apple-Silicon PyTorch >=1.12
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# load & move model
client = ESMC.from_pretrained("esmc_300m").to(device)

# the rest stays the same
protein = ESMProtein(sequence="AAAAA")
protein_tensor = client.encode(protein)
logits_output = client.logits(protein_tensor, LogitsConfig(sequence=True, return_embeddings=True))
print(logits_output.logits, logits_output.embeddings)

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

TypeError: replace() argument 2 must be str, not None

In [2]:
# 1) Make sure embpy is on your PYTHONPATH
import sys
from pathlib import Path

PROJECT_ROOT = Path("/Users/grpinto/Documents/embpy").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 2) Import and configure logging
import logging  # noqa: E402

logging.basicConfig(level=logging.INFO)

# 3) Import your embedder
from embpy.embedder import BioEmbedder  # noqa: E402

# 4) Instantiate the BioEmbedder
#    – For local DNA: pass resolver_backend="local" plus mart/chrom paths
#    – For API: omit mart_file & chromosome_folder (or set resolver_backend="api")
be = BioEmbedder(
    device="auto",
    resolver_backend="local",
    mart_file="/Users/grpinto/Documents/embpy/data/mart_export-5.txt",
    chromosome_folder="/Users/grpinto/Documents/embpy/data/chromosomes",
)

# 5) Pick a model and embed
for model_key in ["enformer_human_rough", "borzoi_v0"]:
    try:
        emb = be.embed_gene(
            identifier="TP53",
            model=model_key,
            id_type="symbol",  # or "ensembl_id", "uniprot_id"
            organism="human",  # used by API resolvers
            pooling_strategy="mean",  # or "max"
        )
        print(f"{model_key:20} → embedding shape:", emb.shape)
        print(emb)
    except Exception as e:  # noqa: BLE001
        print(f"{model_key:20} → ERROR:", e)

INFO:root:MPS device found, using Apple Silicon GPU.
INFO:root:GeneResolver initialized.
INFO:root:pyensembl found. Downloading and indexing if necessary...
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle
INFO:root:pyensembl data ready.
INFO:root:BioEmbedder initialized on device mps, backend=local
INFO:root:Available models: ['bert_base_uncased', 'borzoi_v0', 'borzoi_v1', 'chemberta_zinc_v1', 'enformer_human_rough', 'esm2_150M', 'esm2_35M', 'esm2_3B', 'esm2_650M', 'esm2_8M', 'minilm_l6_v2', 'molformer_base']
INFO:root:Loading model 'enformer_human_rough' onto de

enformer_human_rough → ERROR: Embedding failed for TP53


INFO:root:Borzoi 'johahi/borzoi-replicate-0' loaded on mps (trunk_dim=1536).
INFO:root:Model 'borzoi_v0' loaded successfully.
/Users/grpinto/Documents/embpy/src/embpy/resources/gene_resolver.py:68: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(self.mart_file)


Running Borzoi model …
tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]], device='mps:0')
torch.Size([1, 1536, 6144])
borzoi_v0            → embedding shape: (1536,)
[ 0.10198681  0.18477766 -1.0095096  ... -0.33952937  0.05475771
  0.26471585]


In [1]:
# 1) Ensure embpy is on PYTHONPATH
import sys
from pathlib import Path

PROJECT_ROOT = Path("/Users/grpinto/Documents/embpy").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 2) Set up logging
import logging  # noqa: E402

logging.basicConfig(level=logging.INFO)

# 3) Import and instantiate your embedder in API mode (protein via ESM2 uses online UniProt/MyGene)
from embpy.embedder import BioEmbedder  # noqa: E402

be = BioEmbedder(
    device="auto",
    resolver_backend="api",  # protein lookup via API
)

# 4) Choose an ESM2 model key from your registry
model_key = "esm2_650M"  # or "esm2_35M", "esm2_150M", etc.

# 5) Call embed_gene for a protein—by default takes a gene symbol and returns embedding
emb = be.embed_gene(
    identifier="TP53",  # or any gene symbol / Ensembl ID / UniProt ID
    model=model_key,
    id_type="symbol",  # or "ensembl_id", "uniprot_id"
    organism="human",  # necessary for protein resolution
    pooling_strategy="mean",  # ESM2 wrappers support mean/max
)

print(f"{model_key} embedding shape:", emb.shape)

INFO:root:MPS device found, using Apple Silicon GPU.
INFO:root:GeneResolver initialized.
INFO:root:pyensembl found. Downloading and indexing if necessary...
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle
INFO:root:pyensembl data ready.
INFO:root:BioEmbedder initialized on device mps, backend=api
INFO:root:Available models: ['bert_base_uncased', 'borzoi_v0', 'borzoi_v1', 'chemberta_zinc_v1', 'enformer_human_rough', 'esm2_150M', 'esm2_35M', 'esm2_3B', 'esm2_650M', 'esm2_8M', 'minilm_l6_v2', 'molformer_base']
INFO:root:Loading model 'esm2_650M' onto device 'mps'...

esm2_650M embedding shape: (1280,)


In [5]:
import sys
from pathlib import Path

# 1) Ensure project root is on PYTHONPATH
PROJECT_ROOT = Path("/Users/grpinto/Documents/embpy").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 2) Imports
import logging  # noqa: E402

logging.basicConfig(level=logging.INFO)

from embpy.embedder import BioEmbedder  # noqa: E402

# 3) Instantiate BioEmbedder (local DNA, API for protein/text)
be = BioEmbedder(
    device="auto",
    resolver_backend="api",  # 'local' for DNA FASTA, 'api' for online lookups
    mart_file="/Users/grpinto/Documents/embpy/data/mart_export-5.txt",
    chromosome_folder="/Users/grpinto/Documents/embpy/data/chromosomes",
)

# 4) Define a list of gene identifiers to batch-test
identifiers = ["TP53", "BRCA1", "EGFR", "MYC", "INVALID"]

# 5) Choose a model for DNA, protein, or text
model_key = "esm2_650M"  # protein model
# model_key = "enformer_human_rough"   # DNA model
# model_key = "minilm_l6_v2"           # text model

# 6) Batch embed and inspect results
embeddings = be.embed_genes_batch(
    identifiers=identifiers,
    model=model_key,
    id_type="symbol",
    organism="human",
    pooling_strategy="mean",
)

# 7) Display results
for iden, emb in zip(identifiers, embeddings, strict=False):
    if emb is None:
        print(f"{iden}: ❌ failed to embed")
    else:
        print(f"{iden}: ✅ embedded, shape = {emb.shape}")

INFO:root:MPS device found, using Apple Silicon GPU.
INFO:root:GeneResolver initialized.
INFO:root:pyensembl found. Downloading and indexing if necessary...
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle
INFO:root:pyensembl data ready.
INFO:root:BioEmbedder initialized on device mps, backend=api
INFO:root:Available models: ['bert_base_uncased', 'borzoi_v0', 'borzoi_v1', 'chemberta_zinc_v1', 'enformer_human_rough', 'esm2_150M', 'esm2_35M', 'esm2_3B', 'esm2_650M', 'esm2_8M', 'minilm_l6_v2', 'molformer_base']
INFO:root:Loading model 'esm2_650M' onto device 'mps'...

TP53: ✅ embedded, shape = (1280,)
BRCA1: ✅ embedded, shape = (1280,)
EGFR: ✅ embedded, shape = (1280,)
MYC: ✅ embedded, shape = (1280,)
INVALID: ❌ failed to embed


In [2]:
# 1) Ensure embpy is on PYTHONPATH
import sys
from pathlib import Path

PROJECT_ROOT = Path("/Users/grpinto/Documents/embpy/src").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 2) Imports & logging
import logging  # noqa: E402

logging.basicConfig(level=logging.INFO)
from embpy.embedder import BioEmbedder  # noqa: E402

# 3) Instantiate BioEmbedder
be = BioEmbedder(
    device="auto",
    resolver_backend="api",  # use "local" if you want DNA from FASTAs
    mart_file="/Users/grpinto/Documents/embpy/data/mart_export-5.txt",
    chromosome_folder="/Users/grpinto/Documents/embpy/data/chromosomes",
)

# 4) Prepare a small batch of gene symbols
genes = ["TP53", "BRCA1", "EGFR", "INVALID"]

# 5) Test each model
for model_key in ["enformer_human_rough", "borzoi_v0", "esm2_650M"]:
    print(f"\n=== Embedding with {model_key} ===")
    embs = be.embed_genes_batch(
        identifiers=genes,
        model=model_key,
        id_type="symbol",
        organism="human",
        pooling_strategy="mean",
    )
    for gene, emb in zip(genes, embs, strict=False):
        status = f"shape={emb.shape}" if emb is not None else "FAILED"
        print(f"  {gene:8} → {status}")

INFO:root:MPS device found, using Apple Silicon GPU.
INFO:root:GeneResolver initialized.
INFO:root:pyensembl found. Downloading and indexing if necessary...
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle
INFO:root:pyensembl data ready.
INFO:root:BioEmbedder initialized on device mps, backend=api
INFO:root:Available models: ['bert_base_uncased', 'borzoi_v0', 'borzoi_v1', 'chemberta_zinc_v1', 'enformer_human_rough', 'esm2_150M', 'esm2_35M', 'esm2_3B', 'esm2_650M', 'esm2_8M', 'minilm_l6_v2', 'molformer_base']
INFO:root:Loading model 'enformer_human_rough' onto devi


=== Embedding with enformer_human_rough ===


INFO:root:Enformer 'EleutherAI/enformer-official-rough' loaded on mps.
INFO:root:Model 'enformer_human_rough' loaded successfully.
INFO:root:Batch: 4 items for 'enformer_human_rough' (dna)...
ERROR:root:Error fetching DNA sequence from Ensembl for 'INVALID': 400 Client Error: Bad Request for url: https://rest.ensembl.org/lookup/symbol/human/INVALID?expand=1
INFO:root:Loading model 'borzoi_v0' onto device 'mps'...
INFO:root:Loading Borzoi 'johahi/borzoi-replicate-0' …


  TP53     → shape=(3072,)
  BRCA1    → shape=(3072,)
  EGFR     → shape=(3072,)
  INVALID  → FAILED

=== Embedding with borzoi_v0 ===


INFO:root:Borzoi 'johahi/borzoi-replicate-0' loaded on mps (trunk_dim=1536).
INFO:root:Model 'borzoi_v0' loaded successfully.
INFO:root:Batch: 4 items for 'borzoi_v0' (dna)...
ERROR:root:Error fetching DNA sequence from Ensembl for 'INVALID': 400 Client Error: Bad Request for url: https://rest.ensembl.org/lookup/symbol/human/INVALID?expand=1
INFO:root:Loading model 'esm2_650M' onto device 'mps'...
INFO:root:Loading ESM2 model 'facebook/esm2_t33_650M_UR50D'...


  TP53     → shape=(1536,)
  BRCA1    → shape=(1536,)
  EGFR     → shape=(1536,)
  INVALID  → FAILED

=== Embedding with esm2_650M ===


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
INFO:root:ESM2 model loaded successfully.
INFO:root:Model 'esm2_650M' loaded successfully.
INFO:root:Batch: 4 items for 'esm2_650M' (protein)...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


  TP53     → shape=(1280,)
  BRCA1    → shape=(1280,)
  EGFR     → shape=(1280,)
  INVALID  → FAILED


# Embed Molecules

In [1]:
import sys
from pathlib import Path

# 1) Ensure embpy package is on PYTHONPATH
PROJECT_ROOT = Path("/Users/grpinto/Documents/embpy/src").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 2) Imports and logging
import logging  # noqa: E402

logging.basicConfig(level=logging.INFO)
from embpy.embedder import BioEmbedder  # noqa: E402

# 3) Instantiate the BioEmbedder
be = BioEmbedder(
    device="auto",
    resolver_backend="api",  # molecule embedding does not use gene resolver
    mart_file="/Users/grpinto/Documents/embpy/data/mart_export-5.txt",
    chromosome_folder="/Users/grpinto/Documents/embpy/data/chromosomes",
)

# 4) Single-molecule test
smiles = "CCO"  # ethanol
for model_key in ["chemberta_zinc_v1", "molformer_base"]:
    try:
        emb = be.embed_molecule(identifier=smiles, model=model_key, pooling_strategy="mean")
        print(f"{model_key}: succeeded, shape = {emb.shape}")
    except Exception as e:  # noqa: BLE001
        print(f"{model_key}: ERROR → {e}")

# 5) Batch-molecule test
smiles_list = ["CCO", "c1ccccc1", "INVALID_SMILES"]
print("\nBatch embedding:")
batch_embs = be.embed_molecules_batch(identifiers=smiles_list, model="chemberta_zinc_v1", pooling_strategy="mean")
for sm, out in zip(smiles_list, batch_embs):  # noqa: B905
    status = f"shape = {out.shape}" if out is not None else "FAILED"
    print(f"{sm:12} → {status}")

INFO:rdkit:Enabling RDKit 2025.03.3 jupyter extensions
INFO:root:MPS device found, using Apple Silicon GPU.
INFO:root:GeneResolver initialized.
INFO:root:pyensembl found. Downloading and indexing if necessary...
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle
INFO:root:pyensembl data ready.
INFO:root:BioEmbedder initialized on device mps, backend=api
INFO:root:Available models: ['bert_base_uncased', 'borzoi_v0', 'borzoi_v1', 'chemberta_zinc_v1', 'enformer_human_rough', 'esm2_150M', 'esm2_35M', 'esm2_3B', 'esm2_650M', 'esm2_8M', 'minilm_l6_v2', 'molformer_base']
I

chemberta_zinc_v1: succeeded, shape = (768,)


INFO:root:MolFormer loaded successfully.
INFO:root:Model 'molformer_base' loaded successfully.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
[15:04:19] SMILES Parse Error: syntax error while parsing: INVALID_SMILES
[15:04:19] SMILES Parse Error: check for mistakes around position 3:
[15:04:19] INVALID_SMILES
[15:04:19] ~~^
[15:04:19] SMILES Parse Error: Failed parsing SMILES 'INVALID_SMILES' for input: 'INVALID_SMILES'
INFO:root:Embedding 2 valid SMILES with model 'chemberta_zinc_v1'
ERROR:root:Batch embedding failed: ChembertaWrapper.embed_batch() missing 1 required positional argument: 'smiles_list'


molformer_base: succeeded, shape = (768,)

Batch embedding:
CCO          → FAILED
c1ccccc1     → FAILED
INVALID_SMILES → FAILED


In [3]:
# 1) Make sure your project is on PYTHONPATH
import sys
from pathlib import Path

PROJECT_ROOT = Path("/Users/grpinto/Documents/embpy/src").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 2) Imports & logging
import logging  # noqa: E402

logging.basicConfig(level=logging.INFO)
from rdkit import Chem  # noqa: E402
from embpy.embedder import BioEmbedder  # noqa: E402

# 3) Quick RDKit sanity check
print("RDKit SMILES validity check:")
for smi in ["CCO", "c1ccccc1", "INVALID_SMILES"]:
    mol = Chem.MolFromSmiles(smi)
    print(f"  {smi:12} →", "valid" if mol else "invalid")
print()

# 4) Instantiate your embedder (molecule models ignore resolver_backend)
be = BioEmbedder(
    device="auto",
    resolver_backend="api",
    mart_file="/Users/grpinto/Documents/embpy/data/mart_export-5.txt",
    chromosome_folder="/Users/grpinto/Documents/embpy/data/chromosomes",
)

# 5) Test single-molecule embedding
print("Single‐molecule embedding:")
for model_key in ["chemberta_zinc_v1", "molformer_base"]:
    try:
        emb = be.embed_molecule("CCO", model=model_key)
        print(f"  {model_key:20} → success, shape = {emb.shape}")
    except Exception as e:  # noqa: BLE001
        print(f"  {model_key:20} → FAILED ({e})")
print()

# 6) Test batch‐molecule embedding
print("Batch‐molecule embedding:")
smiles_list = ["CCO", "c1ccccc1", "INVALID_SMILES"]
batch = be.embed_molecules_batch(smiles_list, model="chemberta_zinc_v1")
for smi, out in zip(smiles_list, batch, strict=False):
    status = f"shape = {out.shape}" if out is not None else "FAILED"
    print(f"  {smi:12} → {status}")

[15:18:40] SMILES Parse Error: syntax error while parsing: INVALID_SMILES
[15:18:40] SMILES Parse Error: check for mistakes around position 3:
[15:18:40] INVALID_SMILES
[15:18:40] ~~^
[15:18:40] SMILES Parse Error: Failed parsing SMILES 'INVALID_SMILES' for input: 'INVALID_SMILES'
INFO:root:MPS device found, using Apple Silicon GPU.
INFO:root:GeneResolver initialized.
INFO:root:pyensembl found. Downloading and indexing if necessary...


RDKit SMILES validity check:
  CCO          → valid
  c1ccccc1     → valid
  INVALID_SMILES → invalid



INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle
INFO:root:pyensembl data ready.
INFO:root:BioEmbedder initialized on device mps, backend=api
INFO:root:Available models: ['bert_base_uncased', 'borzoi_v0', 'borzoi_v1', 'chemberta_zinc_v1', 'enformer_human_rough', 'esm2_150M', 'esm2_35M', 'esm2_3B', 'esm2_650M', 'esm2_8M', 'minilm_l6_v2', 'molformer_base']
INFO:root:Loading model 'chemberta_zinc_v1' onto device 'mps'...
INFO:root:Loading ChemBERTa 'seyonec/ChemBERTa-zinc-base-v1'…


Single‐molecule embedding:


INFO:root:Model 'chemberta_zinc_v1' loaded successfully.
INFO:root:Loading model 'molformer_base' onto device 'mps'...
INFO:root:Loading MolFormer 'ibm/MoLFormer-XL-both-10pct' (trust_remote_code)…


  chemberta_zinc_v1    → success, shape = (768,)


INFO:root:MolFormer loaded successfully.
INFO:root:Model 'molformer_base' loaded successfully.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
[15:18:42] SMILES Parse Error: syntax error while parsing: INVALID_SMILES
[15:18:42] SMILES Parse Error: check for mistakes around position 3:
[15:18:42] INVALID_SMILES
[15:18:42] ~~^
[15:18:42] SMILES Parse Error: Failed parsing SMILES 'INVALID_SMILES' for input: 'INVALID_SMILES'
INFO:root:Embedding 2 valid SMILES with model 'chemberta_zinc_v1'


  molformer_base       → success, shape = (768,)

Batch‐molecule embedding:
  CCO          → shape = (768,)
  c1ccccc1     → shape = (768,)
  INVALID_SMILES → FAILED


In [2]:
print(batch)

[array([ 1.1508173 ,  2.0531797 , -0.7293871 , -2.503035  ,  0.99568206,
       -0.6166463 , -0.2915732 , -0.19517012, -0.03072713,  1.0420717 ,
        0.12325957, -2.704893  ,  1.8571848 , -2.0305023 , -0.14597774,
        0.99710244, -0.7364792 , -0.6306886 ,  0.78062487,  1.5268489 ,
        0.28774536,  0.46004376, -0.5138755 , -0.7089479 ,  0.17330335,
        0.59843314,  0.6865997 , -0.804056  , -0.39541817, -0.49063262,
       -0.2725663 , -0.6594177 ,  0.47795615, -0.08630051, -0.10636435,
       -0.12273664, -0.29431552, -0.4415953 , -1.2753153 ,  0.06288067,
        0.1079899 , -0.76255006, -0.88608384, -0.79767275, -0.34571472,
       -0.22518218,  0.04129774, -0.9702606 , -0.41527703,  1.13531   ,
        0.44931075,  1.3730732 ,  1.24929   , -1.0405701 , -1.1639998 ,
        1.2950426 ,  0.48157582,  0.54860944,  0.87889606,  0.35802528,
       -1.0006365 ,  1.108918  , -0.36523727, -0.32436872, -0.40335882,
        0.99921227, -1.4904591 ,  0.6195121 ,  1.136156  , -0.2

In [2]:
import torch
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

# pick device automatically
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # for Apple-Silicon PyTorch >=1.12
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# load & move model
client = ESMC.from_pretrained("esmc_300m").to(device)

# the rest stays the same
protein = ESMProtein(sequence="AAAAA")
protein_tensor = client.encode(protein)
logits_output = client.logits(protein_tensor, LogitsConfig(sequence=True, return_embeddings=True))
print(logits_output.logits, logits_output.embeddings)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/ictstr01/home/icb/goncalo.pinto/tools/apps/mamba/envs/embpy/lib/python3.12/site-packages/esm/pretrained.py:74: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = to

ForwardTrackData(sequence=tensor([[[-38.2500, -38.0000, -38.0000,  12.5625,  21.6250,  22.2500,  22.0000,
           21.7500,  21.3750,  21.6250,  21.6250,  21.2500,  20.7500,  21.3750,
           21.8750,  20.8750,  20.7500,  20.2500,  20.3750,  20.1250,  21.7500,
           19.7500,  19.7500,  19.2500,  18.3750,   1.1641,  -1.8047,  -4.0312,
          -20.6250, -38.2500, -38.0000, -38.0000, -38.2500, -38.2500, -38.2500,
          -38.2500, -38.2500, -38.0000, -38.2500, -38.0000, -38.2500, -38.2500,
          -38.2500, -38.0000, -38.0000, -38.2500, -38.2500, -38.2500, -38.2500,
          -38.2500, -38.2500, -38.2500, -38.0000, -38.2500, -38.2500, -38.0000,
          -38.2500, -38.0000, -38.2500, -38.0000, -38.2500, -38.2500, -38.2500,
          -38.2500],
         [-40.0000, -40.0000, -40.2500,   5.3438,  20.1250,  20.0000,  18.7500,
           20.6250,  18.7500,  18.3750,  18.6250,  18.5000,  18.2500,  18.0000,
           18.7500,  17.7500,  17.3750,  17.1250,  17.7500,  16.8750,  22

In [16]:
import torch
from embpy.models.protein_models import ESM2Wrapper

# 1) pick your device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# 2) instantiate the wrapper
#    - use backend="hf" to call Hugging-Face ESM2
#    - use backend="esmc" to call the ESMC SDK (requires esm installed)
wrapper = ESM2Wrapper(
    model_path_or_name="facebook/esm2_t33_650M_UR50D",  # ignored if backend="esmc"
    backend="hf",                                    # or "esmc"
)

# 3) load weights (and tokenizer or ESMC client) onto your device
wrapper.load(device)

# 4) embed a protein sequence
sequence = "MEEPQSDPSV"
embedding = wrapper.embed(
    input=sequence,
    pooling_strategy="mean",   # one of "mean", "max", or "cls"
    target_layer=None          # only used for HF-backend if you want a middle layer
)

print(f"Sequence length: {len(sequence)}")
print(f"Embedding shape:   {embedding.shape}")
print(embedding[:5], "...")   # show first 5 dims


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Sequence length: 10
Embedding shape:   (1280,)
[0.05464638 0.07903737 0.09929101 0.12977666 0.04402417] ...


In [3]:
from embpy.models.protein_models import ESMCWrapper

wrapper = ESMCWrapper()
wrapper.load(device)
vec = wrapper.embed("MEEPQSDPSV", pooling_strategy="mean")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/ictstr01/home/icb/goncalo.pinto/tools/apps/mamba/envs/embpy/lib/python3.12/site-packages/esm/pretrained.py:74: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = to

In [3]:
%cd /lustre/groups/ml01/workspace/goncalo.pinto/embpy

/ictstr01/groups/ml01/workspace/goncalo.pinto/embpy


In [2]:
!ls

'=1.15'						 slurm_error_34902713.txt
 abind						 slurm_error_34902833.txt
 activations_debug.log				 slurm_error_34902834.txt
 askpass					 slurm_error_34902892.txt
 Attempting_stuff.ipynb				 slurm_error_34943393.txt
 base64enc					 slurm_error_34943396.txt
 BH						 slurm_error_34943397.txt
 bioconductor_docker_3.18-R-4.3.2.sif		 slurm_error_35087604.txt
 bitops						 slurm_error_35087605.txt
 brew						 slurm_error_35087606.txt
 brio						 slurm_error_35087815.txt
 bslib						 slurm_error_35087818.txt
 cachem						 slurm_error_35541823.txt
 callr						 slurm_error_35541824.txt
 caTools					 slurm_error_35541825.txt
 ccle_protein_esm2_650M_mean.csv		 slurm_error_35541843.txt
 cli						 slurm_error_35542740.txt
 clipr						 slurm_error_35542742.txt
 colorspace					 slurm_error_35543797.txt
 commonmark					 slurm_error_35543798.txt
 cowplot					 slurm_error_35543802.txt
 cpp11						 slurm_error_35543804.txt
 crayon						 slurm_error_35583770.txt
 crosstalk					 slurm_error_35583853.

In [4]:
!pip install .

Processing /ictstr01/groups/ml01/workspace/goncalo.pinto/embpy
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for embpy: filename=embpy-0.0.1-py3-none-any.whl size=34305 sha256=03ed0395ed196ee38fed459544d3b1c577cdfdfd2b492137634ac4ddcbeca226
  Stored in directory: /tmp/pip-ephem-wheel-cache-2lz1nonp/wheels/49/ea/da/0fc126f0770df88edfb19e068467e8d252030c96d798a2b20c
Successfully built embpy
  Attempting uninstall: embpy
    Found existing installation: embpy 0.0.1
    Uninstalling embpy-0.0.1:
      Successfully uninstalled embpy-0.0.1


In [1]:
import torch
from embpy.embedder import BioEmbedder

# pick your device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# initialize your BioEmbedder
be = BioEmbedder(device=device)

# protein sequence to embed
seq = "MEEPQSDPSV"

for esm_model in ["esmc_300m", "esmc_600m"]:
    print(f"\n--- testing {esm_model} ---")
    # grab the wrapped model (will load on first call)
    wrapper = be._get_model(esm_model)
    wrapper.load(device)   # should be a no-op if already loaded
    
    # embed directly via the wrapper
    emb = wrapper.embed(seq, pooling_strategy="mean")
    print(f"{esm_model} → embedding shape: {emb.shape}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/ictstr01/home/icb/goncalo.pinto/tools/apps/mamba/envs/embpy/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/ictstr01/home/icb/goncalo.pinto/tools/apps/mamba/envs/embpy/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/ictstr01/home/icb/goncalo.pinto/tools/apps/mamba/envs/embpy/lib

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/ictstr01/home/icb/goncalo.pinto/tools/apps/mamba/envs/embpy/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/ictstr01/home/icb/goncalo.pinto/tools/apps/mamba/envs/embpy/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/ictstr01/home/icb/goncalo.pinto/tools/apps/mamba/envs/embpy/lib

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/ictstr01/home/icb/goncalo.pinto/tools/apps/mamba/envs/embpy/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/ictstr01/home/icb/goncalo.pinto/tools/apps/mamba/envs/embpy/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/ictstr01/home/icb/goncalo.pinto/tools/apps/mamba/envs/embpy/lib

...

config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

esmc_600m_2024_12_v0.pth:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

/ictstr01/home/icb/goncalo.pinto/tools/apps/mamba/envs/embpy/lib/python3.12/site-packages/esm/pretrained.py:92: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = to

esmc_600m → embedding shape: (1152,)
